In [ ]:
# import Pkg; Pkg.add(["Lux", "DifferentialEquations", "SciMLSensitivity", "Optimization", "OptimizationOptimisers", "Statistics", "DataFrames", "CSV", "Plots", "Random", "ComponentArrays"])

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\ADMIN\Downloads\Neural_Spiking_Dynamics\Project.toml`
    Manifest No packages added to or removed from `C:\Users\ADMIN\Downloads\Neural_Spiking_Dynamics\Manifest.toml`


In [1]:
using Lux, DifferentialEquations, SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics,DataFrames,CSV,Plots,Random,ComponentArrays

In [3]:
data = "notebooks/1_data_generation/0_noise/HH_4D_data.csv"

"notebooks/1_data_generation/0_noise/HH_4D_data.csv"

In [ ]:
file_path = raw"c:/Users/nirbh/Neural_Spiking_Dynamics/notebooks/1_data_generation/0_noise/HH_4D_data.csv"
HH_data = CSV.read(file_path, DataFrame)

ArgumentError: ArgumentError: "c:/Users/nirbh/Neural_Spiking_Dynamics/notebooks/1_data_generation/0_noise/HH_4D_data.csv" is not a valid file or doesn't exist

In [5]:
df_ordered = HH_data[:, [:t, :V, :n, :m, :h]]

UndefVarError: UndefVarError: `HH_data` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [6]:
t_train =  Float32.(df_ordered.t)
z_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')

UndefVarError: UndefVarError: `df_ordered` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [7]:
using Plots
theme(:dark)
# Create 4 subplots in a 2x2 grid
p1 = plot(df_ordered.t, df_ordered.V, title="Voltage (V)", ylabel="mV")
p2 = plot(df_ordered.t, df_ordered.n, title="n-gate", color=:green)
p3 = plot(df_ordered.t, df_ordered.m, title="m-gate", color=:red)
p4 = plot(df_ordered.t, df_ordered.h, title="h-gate", color=:cyan)

plot(p1, p2, p3, p4, layout=(2, 2), legend=false)


UndefVarError: UndefVarError: `df_ordered` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [8]:
rng = Random.default_rng()


TaskLocalRNG()

### Scale_Factor

In [9]:
function calculate_scale_factors(z_train,t_train)
          # det the time step (dt)
          dt = diff(t_train) # ( t_i+1 - t_i )

          dz = diff(z_train, dims=2)
          # calculate derivatives ( Finite difference)
          derivatives = dz ./ dt'
          ## result is a 4-element vector: [s_v, s_n, s_m, s_h]
          s_factors = std(derivatives, dims=2)
          


          return s_factors # [s_v, s_n, s_m, s_h]

end

calculate_scale_factors (generic function with 1 method)

In [10]:
# Run this before your optimization loop (Compute on CPU, move to GPU)
scale_factors = calculate_scale_factors(z_train_cpu, t_train_cpu) |> gpu_dev


UndefVarError: UndefVarError: `z_train` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
nn = Lux.Chain(
    Lux.Dense(5 => 64, Lux.tanh),
    Lux.LayerNorm(64),
    Lux.Dense(64 => 64, Lux.tanh),
    Lux.LayerNorm(64),
    Lux.Dense(64 => 4)
)
rng = Random.default_rng()
ps, st = Lux.setup(rng, nn)

# --- Move Network to GPU ---
ps_gpu = ps |> gpu_dev
st_gpu = st |> gpu_dev

# Flatten parameters for the optimizer
p_flat, reconstruct = Optimisers.destructure(ps_gpu)
p_initial = Float32.(p_flat)


ComponentVector{Float32}(layer_1 = (weight = Float32[-0.0076249023 -1.099113 … -0.90581286 -1.1629107; 1.0789473 -0.8968392 … 0.830733 0.81494856; … ; -0.702894 0.81198984 … -1.1788998 0.13746062; 0.2314203 0.24849749 … 0.3079428 -1.1398731], bias = Float32[-0.38768768, 0.0889134, 0.076824, -0.33325768, -0.11932855, 0.2617155, -0.3177188, -0.22066034, 0.3156114, -0.42457753  …  -0.045831166, -0.33149186, -0.045571484, -0.11859087, -0.13794537, 0.16899328, 0.42464224, 0.42413947, 0.38225025, 0.013726245]), layer_2 = (bias = Float32[0.0; 0.0; … ; 0.0; 0.0;;], scale = Float32[1.0; 1.0; … ; 1.0; 1.0;;]), layer_3 = (weight = Float32[-0.06792476 -0.31948954 … -0.15848678 -0.34727472; 0.09842732 -0.13995436 … 0.114625536 0.24624085; … ; -0.2920922 -0.021089686 … 0.04603889 -0.28871903; -0.16033703 0.22710617 … 0.2995993 0.21784401], bias = Float32[0.097737625, 0.01473777, 0.062784255, -0.063441396, 0.022137627, 0.0049549043, -0.019841447, -0.066644445, -0.0861647, 0.09397127  …  0.10703334, 0

In [12]:
function hh_equations(u)
    # Extract the 4 state variables from the input matrix
    # Using u[1:1, :] instead of u[1, :] keeps them as 2D arrays (1 x N matrices),
    # which makes the math operations play nicely with Zygote (the automatic differentiator)
    V = u[1:1, :] 
    n = u[2:2, :]
    m = u[3:3, :]
    h = u[4:4, :]
    
    # 1. Physical Parameters 
    I_ext = 10.0f0
    g_Na  = 120.0f0
    g_K   = 36.0f0
    g_L   = 0.3f0
    E_Na  = 50.0f0
    E_K   = -77.0f0
    E_L   = -54.4f0
    C_m   = 1.0f0

    # 2. Compute Ionic Currents using array broadcasting (.)
    I_Na = g_Na .* (m.^3) .* h .* (V .- E_Na)
    I_K  = g_K .* (n.^4) .* (V .- E_K)
    I_L  = g_L .* (V .- E_L)
    
    # 3. Compute True Derivatives
    # Membrane potential (dV/dt)
    dV = (I_ext .- I_Na .- I_K .- I_L) ./ C_m
    
    # Gating variables (dn/dt, dm/dt, dh/dt)
    # Using the alpha and beta rate functions you defined earlier
    dn = α_n.(V) .* (1.0f0 .- n) .- β_n.(V) .* n
    dm = α_m.(V) .* (1.0f0 .- m) .- β_m.(V) .* m
    dh = α_h.(V) .* (1.0f0 .- h) .- β_h.(V) .* h
    
    # 4. Stack the derivatives back into a 4 x N matrix
    return vcat(dV, dn, dm, dh)
end


hh_equations (generic function with 1 method)

### Loss Function ( total_loss = loss_data + Physics_loss)


In [13]:
function pyhysics_loss(nn, state_data, time_data, θ, scale_factors, reconstruct)
    ps_current = reconstruct(θ)
    
    # Reshape time_data to 1xN matrix to match GPU dimensions
    nn_inputs = vcat(state_data, reshape(time_data, 1, :))

    # Evaluate neural network on GPU
    pred_derivs, _ = nn(nn_inputs, ps_current, st_gpu)

    # calculate the HH derivatives
    true_dervis = hh_equations(state_data)

    # compute scale-aware residual
    residuals = (pred_derivs .- true_dervis) ./ scale_factors
    
    return mean(residuals.^2)
end


pyhysics_loss (generic function with 1 method)

### The Training Synergy

In [14]:
# Float32 Rate Functions
α_m(V) = 0.1f0 .* (V .+ 40.0f0) ./ (1.0f0 .- exp.(-(V .+ 40.0f0) ./ 10.0f0))
β_m(V) = 4.0f0 .* exp.(-(V .+ 65.0f0) ./ 18.0f0)
α_h(V) = 0.07f0 .* exp.(-(V .+ 65.0f0) ./ 20.0f0)
β_h(V) = 1.0f0 ./ (1.0f0 .+ exp.(-(V .+ 35.0f0) ./ 10.0f0))
α_n(V) = 0.01f0 .* (V .+ 55.0f0) ./ (1.0f0 .- exp.(-(V .+ 55.0f0) ./ 10.0f0))
β_n(V) = 0.125f0 .* exp.(-(V .+ 65.0f0) ./ 80.0f0)

function neural_dynamics(u, p, t)
    ps_structured = reconstruct(p)
    
    # Wrap scalar 't' in a GPU array before concatenating
    t_arr = fill(t, 1) |> gpu_dev
    raw_input = vcat(u, t_arr) 
    
    input_2d = reshape(raw_input, :, 1)
    
    # Evaluate on GPU
    dudt_2d, _ = nn(input_2d, ps_structured, st_gpu) 
    
    return vec(dudt_2d)
end

p = Float32[
    10.0,  # External current
    120.0, # Max Sodium conductance
    36.0,  # Max Potassium conductance
    0.3,   # Leak conductance
    50.0,  # Sodium reversal potential
    -77.0, # Potassium reversal potential
    -54.4, # Leak reversal potential
    1.0    # Membrane capacitance
]

# Move Initial Conditions to GPU
u0_gpu = Float32[-65.0f0, 0.05f0, 0.6f0, 0.32f0] |> gpu_dev
tspan = (0.0f0, 50.0f0)
node_prob = ODEProblem(neural_dynamics, u0_gpu, tspan, p_initial)


ODEProblem with uType Vector{Float32} and tType Float32. In-place: false
Non-trivial mass matrix: false
timespan: (0.0f0, 50.0f0)
u0: 4-element Vector{Float32}:
 -65.0
   0.05
   0.6
   0.32

In [15]:
function predict(θ)
          new_prob = remake(node_prob, p=θ)

          return solve(new_prob, Heun(), 
                 saveat=t_train, 
                 reltol=1e-6, abstol=1e-6,
                 sensealg=InterpolatingAdjoint())
end

predict (generic function with 1 method)

### Defining the loss function


In [16]:
function total_loss(θ, _)
    # 1. Trajectory Data Loss
    sol = predict(θ) # Calls solve(remake(node_prob, p=θ), Heun()...)
    if sol.retcode != ReturnCode.Success
        return Inf # Gracefully handle stiff solver failures
    end
    loss_data = mean(abs2, Array(sol) .- z_train)

    # 2. Physics Residual Loss (Pass current θ)
    loss_phys = pyhysics_loss(nn, z_train, t_train, θ, scale_factors, reconstruct)

    # 3. Final Weighted Loss (λ = 1.5)
    λ = 1.5f0
    return loss_data + λ * loss_phys
end

total_loss (generic function with 1 method)

In [17]:
# 1. Create an empty array to track the loss
loss_history = Float32[]
# 2. Define the callback function to save the loss at each step
callback_fn = function (θ, loss_val)
    push!(loss_history, loss_val)
    
    # Optional: Print the loss every 10 iterations so you aren't staring at a blank screen
    if length(loss_history) % 10 == 0
        println("Iteration: $(length(loss_history)) | Loss: $(loss_val)")
    end
    
    return false # False means "do not halt the optimization"
end

#23 (generic function with 1 method)

In [18]:
# optimization 
optf  = OptimizationFunction(total_loss, Optimization.AutoZygote())
optprob = OptimizationProblem(optf, p_initial)

# 3. Run the optimization, passing our callback_fn
println("Starting Training...")
result = solve(optprob, Adam(0.001), maxiters = 5000, callback = callback_fn)
println("Training Complete!")

Starting Training...


UndefVarError: UndefVarError: `t_train` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [19]:
# 4. Plot the Loss Graph
using Plots
plot(loss_history, 
    title = "PI-NODE Training Loss",
    xlabel = "Epochs (Iterations)", 
    ylabel = "Total Loss", 
    linewidth = 3, 
    color = :orange, # Using your preferred neon aesthetics
    grid = true,
    label = "Loss (Data + Physics)"
)

ArgumentError: ArgumentError: Unknown color: neonorange